# Mapping a species' occurrences — GBIF + OBIS

Two of the biodiversity-cluster backends record *where a species has been observed*: **GBIF** (the global biodiversity record — museums, citizen science, monitoring) and **OBIS** (the marine record). This showcase pulls both for the **loggerhead sea turtle** (*Caretta caretta*) — which is logged both on land (nesting beaches, in GBIF) and at sea (in OBIS) — and maps them together.

Both backends are *vector* and anonymous: each `download()` returns a points `FeatureCollection` in `EPSG:4326`, so they drop straight onto one map.

> **Install.** This notebook needs both occurrence SDKs: `pip install earthlens[gbif,obis]`. No credentials are required — both searches are anonymous.

## Setup

Import `matplotlib` for the map and the unified `EarthLens` entry point.

In [ ]:
import matplotlib.pyplot as plt

from earthlens import EarthLens

## Area and window

The wider Caribbean / north-west Atlantic — a loggerhead stronghold — over a five-year window. A bounded bbox keeps both queries fast.

In [ ]:
bbox_lat = [10.0, 35.0]
bbox_lon = [-90.0, -60.0]
start, end = "2015-01-01", "2020-12-31"
species = "Caretta caretta"  # loggerhead sea turtle

## GBIF occurrences

`variables=["taxon:Caretta caretta"]` resolves the binomial to a GBIF backbone `taxonKey` live, then searches occurrences in the bbox.

In [ ]:
gbif = EarthLens(
    data_source="gbif",
    variables=[f"taxon:{species}"],
    start=start,
    end=end,
    lat_lim=bbox_lat,
    lon_lim=bbox_lon,
    path="out/showcase",
    max_records=1500,
).download()

print(f"GBIF: {len(gbif)} occurrences")

## OBIS occurrences

The marine record for the same species and bbox.

In [ ]:
obis = EarthLens(
    data_source="obis",
    variables=[f"species:{species}"],
    start=start,
    end=end,
    lat_lim=bbox_lat,
    lon_lim=bbox_lon,
    path="out/showcase",
    size=1500,
).download()

print(f"OBIS: {len(obis)} occurrences")

## Map both sources together

GBIF in green, OBIS in blue — the land/coast records and the at-sea records form a fuller picture of the species' range than either source alone.

In [ ]:
ax = gbif.plot(markersize=6, color="darkgreen", label="GBIF")
obis.plot(ax=ax, markersize=6, color="navy", label="OBIS")
ax.set_title("Caretta caretta occurrences — GBIF + OBIS, NW Atlantic")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.legend()
plt.show()

## Licensing

Both sources mix permissive and restrictive licenses; each backend raises a `LicenseWarning` when a result carries a non-commercial (`CC-BY-NC`) record. Honour the per-record obligations before redistributing, and attribute GBIF and OBIS as their data policies require.